This notebook implements gridsearch hyperparameter optimization with cross-validation for classical, non-deep regressors. 

## 0. Mount drive

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

drive_dir

## 1. Import libraries, get dataset

In [ ]:
import os
import random
import pickle
from google.colab import files

import scipy
import scipy.stats
import pandas as pd
import numpy as np

!pip install pyyaml h5py

import sys
sys.path.insert(0, drive_dir)

import utils

#load .csv into a DataFrame
pd.options.display.max_rows = 10
raw_data = pd.read_csv(f'{drive_dir}Ecoli_data.csv', index_col= 0).dropna().reset_index(drop = True)

## 2. Load saved split of working dataset plus static test set.

In [ ]:
iter_ID   = 1 #choose an integer in [1,5] to load different sets
get_saved = True #set to *True* to use saved working/heldout sets; set to *False* to generate new sets

###
iteration = f'iteration_{iter_ID}/'
series_dict = utils.split_seeds(raw_data)

working_set, heldout_set = utils.get_split(series_dict = series_dict, 
                                           series_list = [sr for sr in range(1,57)],
                                           load_saved = get_saved,
                                           train_size = 0.9, #change to get different dataset ratios
                                           path = f'{drive_dir}_saved/{iteration}')

print(f'Working set contains {working_set.shape[0]} sequences for train and validation\nHeldout contains {heldout_set.shape[0]} sequences for testing')

## 3. Define options and search space for *non-deep* regressors

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

# define non-deep regressors and hyperparameters to search
regs = ['rf', 'mlp', 'svr']
embeddings = ['one-hot', \
              '1-mers', \
              '3-mers', '4-mers', '5-mers', '6-mers',\
              '3-count', '4-count','5-count','6-count',\
              'features', 'mixed']

param_grid_rf = {
    'bootstrap': [True],
    'max_depth': [80, 90, 100, 110],
    'max_features': [2, 3],
    'min_samples_leaf': [3, 4, 5],
    'min_samples_split': [8, 10, 12],
    'n_estimators': [100, 200, 300, 1000]
}

param_grid_mlp = {
    'activation'         : ['logistic', 'relu', 'tanh'],
    'hidden_layer_sizes' : [
        (100, 100), (100, 100, 100), (200, 200),
        (200, 200, 200), (250, 250), (250, 250, 250)
    ],
    'max_iter'           : [100, 200, 250] 
}

param_grid_svr = {
    'kernel'  : ['rbf', 'poly', 'linear'],
    'C'       : [1, 10, 20, 30, 40, 50],
    'epsilon' : [0.1, 0.3, 0.5, 0.7, 1.0]
}

## 4. Hyperparameter Optimization

In [ ]:
# start by splitting the working set
# we will only use the val_df for optizing the hyperparameters
_, val_df = utils.get_split(series_dict = utils.split_seeds(working_set), 
                             series_list = [itr for itr in range(1,57)],
                             train_size  = 0.75
                             )

# create syn_dna() objects
emb_dna = utils.syn_dna(val_df.reset_index(drop=True))

#lists to store predictions
rf_lst, mlp_lst, svr_lst= [], [], []
 
for embd_method in embeddings:
    # encode DNA sequences
    embd = utils.get_encoding(emb_dna, embd_method=embd_method)

    # define target distribution
    phenotype = emb_dna.protein

    #define a regressor params list
    for reg in regs:
      if reg == 'rf':
        # Create a base RF model
        rf = RandomForestRegressor()
        # Instantiate the grid search model
        grid_search_rf = GridSearchCV(estimator = rf, param_grid = param_grid_rf, 
                                      cv = 10, n_jobs = -1, verbose = 2)
        grid_search_rf.fit(embd, phenotype)
        rf_lst.append(grid_search_rf.best_params_)

      elif reg == 'mlp':
        # Create a base MLP model
        mlp = MLPRegressor()
        # Instantiate the grid search model
        grid_search_mlp = GridSearchCV(estimator = mlp, param_grid = param_grid_mlp, 
                                      cv = 10, n_jobs = -1, verbose = 2)
        grid_search_mlp.fit(embd, phenotype) 
        mlp_lst.append(grid_search_mlp.best_params_)

      elif reg == 'svr':
        # Create a base SVR model
        svr = SVR()
        # Instantiate the grid search model
        grid_search_svr = GridSearchCV(estimator = svr, param_grid = param_grid_svr, 
                                      cv = 10, n_jobs = -1, verbose = 2)
        grid_search_svr.fit(embd, phenotype)
        svr_lst.append(grid_search_svr.best_params_)  
      print(rf_lst, mlp_lst, svr_lst)